In [1]:
from elasticsearch import Elasticsearch
import json
import random

In [2]:
es = Elasticsearch("http://localhost:9200")
print(es.info())

{'name': '15b0c13d7651', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'uJCfvgfmQSaZcJLqd6DsJQ', 'version': {'number': '8.10.2', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '6d20dd8ce62365be9b1aca96427de4622e970e9e', 'build_date': '2023-09-19T08:16:24.564900370Z', 'build_snapshot': False, 'lucene_version': '9.7.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


In [3]:
res = es.search(
     index="movies_clean",
     query={"match_all": {}}
)

print(json.dumps(res["hits"]["hits"], indent=2))

[
  {
    "_index": "movies_clean",
    "_id": "300168",
    "_score": 1.0,
    "_source": {
      "tagline": "When the Eagle meets the Dragon",
      "genres": [
        "Adventure",
        "History"
      ],
      "recommendations": "17457-348595-18760-10951-45408-35138-11653-18764-361613-11198-25676-550290-19274-272875-21733-18707-362154-299738-251749-151743-10622",
      "id": 300168,
      "production_companies": "Visualizer Film Company-Shanghai Film Group-Jackie & JJ Production-Sparkle Roll Media-Alibaba Pictures Group-Home Media & Entertainment Fund-Tencent Video-China Film & TV Capital-Huayi Brothers Pictures",
      "backdrop_path": "/rnNkUJlo03a8D0utxiOrADstaMT.jpg",
      "popularity": 38.233,
      "log": {
        "file": {
          "path": "/data/movies.csv"
        }
      },
      "original_language": "zh",
      "release_date": "2015-02-19T00:00:00.000Z",
      "credits": "Jackie Chan-John Cusack-Adrien Brody-Sharni Vinson-Kevin Lee-Raiden Integra-Tomer Oz-Alfred Hs

# Contrôle Qualité : movies_raw vs movies_clean

## 1. Nombre de documents

In [4]:
# Comme un df.shape[0] (Le nombre de ligne)
raw_count = es.count(index="movies_raw")["count"]
clean_count = es.count(index="movies_clean")["count"]

print(f"Données Brutes : {raw_count} documents (avec doublons)")
print(f"Données nettoyé : {clean_count} documents")
print(f"Doublons supprimés : {raw_count - clean_count}")

Données Brutes : 769631 documents (avec doublons)
Données nettoyé : 662083 documents
Doublons supprimés : 107548


## 2. Valeurs manquantes dans movies_clean

In [5]:
# Toute les colonnes
fields = [
    "id", "title", "overview", "tagline", "genres", "original_language",
    "status", "release_date", "popularity", "budget", "revenue", "runtime",
    "vote_average", "vote_count", "production_companies", "credits", "keywords",
    "poster_path", "backdrop_path", "recommendations"
]

# Vérification des valeurs manquantes dans chaque colonne

print(f"{'Champ':<25} {'Manquants':>10} {'%':>8}")
print("-" * 45)
for field in fields:
    count = es.count(index="movies_clean", 
                     body={
                         "query": {
                             "bool": {
                                 "must_not": {
                                     "exists": {
                                         "field": field
                                        }
                                    }
                                }
                            }
                        }
                    )["count"]
    print(f"{field:<25} {count:>10} {count / clean_count * 100:>f}%")

Champ                      Manquants        %
---------------------------------------------
id                                 0 0.000000%
title                              4 0.000604%
overview                      114502 17.294206%
tagline                       568952 85.933637%
genres                        212149 32.042659%
original_language                  0 0.000000%
status                             0 0.000000%
release_date                   59459 8.980596%
popularity                         0 0.000000%
budget                             0 0.000000%
revenue                            0 0.000000%
runtime                        38399 5.799726%
vote_average                       0 0.000000%
vote_count                         0 0.000000%
production_companies          369668 55.834087%
credits                       223092 33.695473%
keywords                      480006 72.499369%
poster_path                   197078 29.766359%
backdrop_path                 480295 72.543020%
recomme

## 3. Comparaison d'un document brut vs nettoyé

In [6]:
# Il prend un document aléatoire 
sample_clean = es.search(index="movies_clean", body={
    "query": {"function_score": {"functions": [{"random_score": {"seed": random.randint(1, 999999), "field": "id"}}]}},
    "size": 1
})
doc = sample_clean["hits"]["hits"][0]["_source"]
doc_id = doc["id"]

# Cherche le même document dans movies_raw pour comparer la donnée brute et celle nettoyée
sample_raw = es.search(index="movies_raw", body={
    "query": {"term": {"id": str(doc_id)}}, "size": 1
})

print("BRUTE (movies_raw)")
if sample_raw["hits"]["hits"]:
    raw_doc = sample_raw["hits"]["hits"][0]["_source"]
    for k in ["title", "release_date", "genres", "budget", "vote_average"]:
        print(f"  {k}: {raw_doc.get(k)}")
else:
    print("  Document non trouvé dans movies_raw")

print()

print("NETTOYÉE (movies_clean)")
for k in ["title", "release_date", "genres", "budget", "vote_average"]:
    print(f"  {k}: {doc.get(k)}")

BRUTE (movies_raw)
  title: Upload
  release_date: 2019-09-13
  genres: Horror-Comedy
  budget: 0.0
  vote_average: 0.0

NETTOYÉE (movies_clean)
  title: Upload
  release_date: 2019-09-13T00:00:00.000Z
  genres: ['Horror', 'Comedy']
  budget: 0
  vote_average: 0.0


## 4. Vérification de l'analyzer custom

In [7]:
# On vérifie que l'analyzer custom est bien appliqué sur movies_clean.
# On envoie une phrase test et on observe les tokens produits :
# - "The" supprimé 
# - "Amazing" → "amaz" 
# - "Spider-Man" → "spider", "man" 
# - "returns" → "return" 
result = es.indices.analyze(
    index="movies_clean",
    body={"analyzer": "movies_text_analyzer", "text": "The Amazing Spider-Man returns"}
)
tokens = [t["token"] for t in result["tokens"]]
print("Tokens générés par movies_text_analyzer :")
print(tokens)

Tokens générés par movies_text_analyzer :
['amaz', 'spider', 'man', 'return']
